<a href="https://colab.research.google.com/github/haskinse/bee2041_empirical_project/blob/main/source_code/03_database_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
from google.colab import drive # connect to google drive
drive.mount("/content/drive")

project_path = "/content/drive/MyDrive/bee2041_empirical_project" # define project path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import pandas as pd # library for data manipulation and tables

In [46]:
artist_data = pd.read_csv(f"{project_path}/data/clean/artist_data.csv") # read clean csv data into a data frame
spotify_data = pd.read_csv(f"{project_path}/data/raw/spotify_data.csv") # read clean csv data into a data frame
track_r_data = pd.read_csv(f"{project_path}/data/clean/track_data.csv") # read clean csv data into a data frame
# this version of tracks will be rational

In [48]:
track_artists_rows = [] # creat list to store track to artist relationships (for many to many relationship)

for i in range(len(track_r_data)): # for as many tracks as there are
  track_id = track_r_data.loc[i, "track_id"] # extract track ID for the current row

  artists = spotify_data.loc[i, "artist_name"].split(",") # split comma separated artist names into a list

  for artist in artists: # for each artist
    artist = artist.strip() # remove extra whitespace from artist name

    artist_id = artist_data.loc[artist_data["artist_name"] == artist, "artist_id"].iloc[0] # look up matching artist ID from artist table

    track_artists_rows.append({"track_id": track_id, "artist_id": artist_id}) # add relationship between track and artist

track_r_data = track_r_data.drop(columns = "artist_name")

track_r_data.to_csv(f"{project_path}/data/clean/track_data.csv", index = False) # save raw data to drive

track_artists = pd.DataFrame(track_artists_rows) # convert list of relationships into a DataFrame

track_artists.to_csv(f"{project_path}/data/clean/track_artists.csv", index = False) # save raw data to drive

track_artists.head() # view first five rows

,track_id,artist_id
0,1,23
1,2,23
2,3,23
3,4,23
4,5,23


In [49]:
import sqlite3 # ibrary used to create and interact with SQLite databases

In [54]:
conn = sqlite3.connect(f"{project_path}/data/clean/taylor_swift.db") # make a connection to SQLite database (creates file if it does not exist)
conn.execute("PRAGMA foreign_keys = ON;") # respect relationships between tables

In [51]:
# the layout of the database with all of the tables included
schema = """
DROP TABLE IF EXISTS track_artists;
DROP TABLE IF EXISTS tracks;
DROP TABLE IF EXISTS albums;
DROP TABLE IF EXISTS artists;

CREATE TABLE artists (
    artist_id     INTEGER PRIMARY KEY,
    artist_name   TEXT NOT NULL UNIQUE,
    listeners     INTEGER,
    playcount     INTEGER
);

CREATE TABLE albums (
    album_id      INTEGER PRIMARY KEY,
    album_name    TEXT NOT NULL UNIQUE,
    us_peak       INTEGER,
    uk_peak       INTEGER,
    release_date  TEXT,
    label         TEXT,
    us_sales      INTEGER,
    uk_sales      INTEGER,
    riaa          TEXT,
    riaa_units    INTEGER,
    bpi           TEXT,
    bpi_units     INTEGER
);

CREATE TABLE tracks (
    track_id           INTEGER PRIMARY KEY,
    track_name         TEXT NOT NULL,
    album_id           INTEGER NOT NULL,
    release_date       TEXT,
    track_number       INTEGER,
    explicit           BOOLEAN,
    tempo              REAL,
    energy             REAL,
    danceability       REAL,
    duration_min       REAL,
    time_signature     TEXT,
    valence            REAL,
    acousticness       REAL,
    instrumentalness   REAL,
    liveness           REAL,
    loudness           REAL,
    speechiness        REAL,
    full_key           TEXT,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);

CREATE TABLE track_artists (
    track_id    INTEGER,
    artist_id   INTEGER,
    PRIMARY KEY (track_id, artist_id),
    FOREIGN KEY (track_id) REFERENCES tracks(track_id),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);
"""

In [52]:
# reuploaded even previously loaded data to ensure consistency
album_data = pd.read_csv(f"{project_path}/data/clean/album_data.csv")
artist_data = pd.read_csv(f"{project_path}/data/clean/artist_data.csv")
track_r_artist = pd.read_csv(f"{project_path}/data/clean/track_artists.csv")
track_data = pd.read_csv(f"{project_path}/data/clean/track_data.csv")

track_r_data.head()

,album_id,release_date,track_name,track_number,explicit,tempo,energy,danceability,duration_min,time_signature,valence,acousticness,instrumentalness,liveness,loudness,speechiness,full_key,track_id
0,1,2006-10-24,tim mcgraw,1,False,76.009,49.1,58.0,3.86845,4/4,42.5,57.5,0.0,12.10,-6.462,2.51,C Major,1
1,1,2006-10-24,picture to burn,2,False,105.586,87.7,65.8,2.88445,4/4,82.1,17.3,0.0,9.62,-2.098,3.23,G Major,2
2,1,2006-10-24,teardrops on my guitar radio single remix,3,False,99.953,41.7,62.1,3.38400,4/4,28.9,28.8,0.0,11.90,-6.941,2.31,Bb Major,3
3,1,2006-10-24,a place in this world,4,False,115.028,77.7,57.6,3.32000,4/4,42.8,5.1,0.0,32.00,-2.881,3.24,A Major,4
4,1,2006-10-24,cold as you,5,False,175.558,48.2,41.8,3.98355,4/4,26.1,21.7,0.0,12.30,-5.769,2.66,F Major,5


In [53]:
conn.executescript(schema) # execute SQL schema to create database tables and structure

artist_data.to_sql("artists", conn, if_exists = "append", index = False) # insert artist data into artists table

album_data.to_sql("albums", conn, if_exists = "append", index = False) # insert album data into albums table

track_r_data.to_sql("tracks", conn, if_exists = "append", index = False) # insert track data into tracks table

# insert track to artist relationships into relational table (many-to-many relationship)
track_artists.to_sql("track_artists", conn, if_exists = "append", index = False)

conn.close() # close database connection